# 🏭 Production Agents

## Production Requirements
A production agent needs:
- **Error handling**: graceful failure recovery
- **Timeout handling**: don't hang forever
- **Rate limiting**: respect API limits
- **Monitoring**: observe what's happening
- **Logging**: debug production issues
- **Cost tracking**: control LLM spend
- **Security**: prevent prompt injection

## Production Checklist
- [ ] All nodes have try/except
- [ ] Max iterations limit set
- [ ] LangSmith tracing enabled
- [ ] API keys in environment variables
- [ ] Fallback LLMs configured
- [ ] Rate limiter implemented
- [ ] Input validation / guardrails
- [ ] Output validation


In [ ]:
# ── Production-Ready Agent Pattern ─────────────────────────────────────
import os
import time
import logging
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class ProductionState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    error_count: int
    max_errors: int
    start_time: float
    timeout_seconds: float

# Production LLM: primary + fallback + retry
primary_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
fallback_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)  # same model as backup demo
production_llm = primary_llm.with_fallbacks([fallback_llm]).with_retry(stop_after_attempt=3)

@tool
def safe_calculate(expression: str) -> str:
    'Safely evaluate a math expression. Input: python math expression.'
    try:
        # Whitelist only safe operations
        allowed = set('0123456789+-*/.()')
        if not all(c in allowed or c.isspace() for c in expression):
            return 'Error: expression contains unsafe characters'
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f'Error: {e}'

def agent_with_error_handling(state: ProductionState) -> dict:
    # Check timeout
    elapsed = time.time() - state['start_time']
    if elapsed > state['timeout_seconds']:
        logger.error(f'Agent timeout after {elapsed:.1f}s')
        return {'messages': [SystemMessage(content='TIMEOUT: Agent took too long')]}

    try:
        response = production_llm.bind_tools([safe_calculate]).invoke(state['messages'])
        logger.info(f'Agent response: {str(response.content)[:50]}...')
        return {'messages': [response]}
    except Exception as e:
        logger.error(f'Agent error: {e}')
        new_error_count = state['error_count'] + 1
        if new_error_count >= state['max_errors']:
            return {'messages': [SystemMessage(content=f'FATAL: {new_error_count} errors')], 'error_count': new_error_count}
        return {'error_count': new_error_count}

print('Production agent pattern ready.')
print('Key features: timeout, error counting, fallbacks, retry, input sanitization')

## 🎯 Interview Questions

1. **What are the minimum requirements for a production agent?**
2. **How do you implement timeouts in LangGraph?**
3. **How do you prevent prompt injection in production?**
4. **How would you monitor agent cost in production?**
5. **What is the graceful degradation pattern?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Production Agents — Battle-Hardened Patterns**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
